# 03c · Expunerea la vânt
Din ERA5-Land orar: direcțiile vântului în orele de vară 2024 cu peste 30 °C și fără ploaie. Pentru fiecare parcelă: ce parte din marginea din amonte **nu** are un element lemnos (Small Woody Features sau Woody Vegetation Layer 2021) până la 20H = 300 m.

- Rezultate: `data/out/wind_rose.parquet` și `data/out/parcel_wind.parquet` (`wind_exposure`: 0 = adăpostită, 1 = complet expusă).
- E un indicator de expunere, nu o măsurătoare a efectului perdelelor.

In [1]:
import os, sys
if not os.path.exists('gisday_config.py') and os.path.exists('../gisday_config.py'):
    os.chdir('..')
sys.path.insert(0, os.getcwd())
from gisday_config import *
from gisday_lib import wind_rose, wind_exposure
import numpy as np, pandas as pd, geopandas as gpd, rasterio, ee
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling
ee.Initialize(project=GEE_PROJECT)
BBOX = ee.Geometry.Rectangle(list(AOI_BBOX))
RO = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Romania'))
AOI = BBOX.intersection(RO.geometry(), ee.ErrorMargin(10))
parcels = gpd.read_parquet(OUT / 'parcels.parquet')
print(len(parcels), 'parcele')

17466 parcele


In [2]:
# Orele de vară 2024, medie pe zonă (ERA5-Land are ~9 km, deci zona are puține celule)
hourly = (ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY').filterDate(f'{YEAR}-06-01', f'{YEAR}-09-01')
          .select(['temperature_2m', 'u_component_of_wind_10m', 'v_component_of_wind_10m', 'total_precipitation_hourly']))
print('benzi:', hourly.first().bandNames().getInfo(), '| ore:', hourly.size().getInfo())
fc = hourly.map(lambda i: ee.Feature(None, i.reduceRegion(ee.Reducer.mean(), AOI, 11000)).set('time', i.date().format('YYYY-MM-dd HH:mm')))
W = pd.DataFrame([f['properties'] for f in fc.getInfo()['features']])
hot_dry = W[(W.temperature_2m > 303.15) & (W.total_precipitation_hourly < 1e-4)]
print('ore peste 30 °C și fără ploaie:', len(hot_dry), 'din', len(W))
rose = wind_rose(hot_dry.u_component_of_wind_10m.to_numpy(), hot_dry.v_component_of_wind_10m.to_numpy())
rose.rename('share').rename_axis('azimuth').reset_index().to_parquet(OUT / 'wind_rose.parquet')
print(rose.round(3).to_string())
main = rose[rose >= 0.05]
print('direcții folosite (>= 5% din ore):', main.round(3).to_dict())

benzi: ['temperature_2m', 'u_component_of_wind_10m', 'v_component_of_wind_10m', 'total_precipitation_hourly'] | ore: 2208


ore peste 30 °C și fără ploaie: 655 din 2208
0.0      0.023
22.5     0.027
45.0     0.035
67.5     0.084
90.0     0.238
112.5    0.253
135.0    0.093
157.5    0.044
180.0    0.021
202.5    0.005
225.0    0.009
247.5    0.014
270.0    0.040
292.5    0.075
315.0    0.021
337.5    0.017
direcții folosite (>= 5% din ore): {67.5: 0.084, 90.0: 0.238, 112.5: 0.253, 135.0: 0.093, 292.5: 0.075}


In [3]:
# Masca elementelor lemnoase (SWF + WVL 2021), adusă în EPSG:3844 la 5 m pe extinderea parcelelor
minx, miny, maxx, maxy = parcels.total_bounds
RES, M = 5.0, 500.0
width, height = int((maxx - minx + 2 * M) / RES), int((maxy - miny + 2 * M) / RES)
transform = from_origin(minx - M, maxy + M, RES, RES)
woody = np.zeros((height, width), dtype=bool)
tifs = sorted(p for p in (RAW / 'slf').rglob('*.tif'))
for p in tifs:
    with rasterio.open(p) as src:
        band = np.zeros((height, width), dtype='uint8')
        reproject(rasterio.band(src, 1), band, dst_transform=transform, dst_crs=CRS, resampling=Resampling.nearest)
    woody |= band == 1
    print(p.name, 'pixeli lemnoși cumulați:', int(woody.sum()))
print(f'element lemnos: {woody.mean():.1%} din fereastră')

SWF_2021_005m_eu_03035_V01_R02.tif pixeli lemnoși cumulați: 2739987


WVL_2021_005m_03035_V01_R02.tif pixeli lemnoși cumulați: 8670884
element lemnos: 10.9% din fereastră


In [4]:
X = parcels.copy()
X['wind_exposure'] = wind_exposure(X, woody, transform, list(main.index), list(main.values))
X[['parcel_id', 'wind_exposure']].to_parquet(OUT / 'parcel_wind.parquet')
assert X.wind_exposure.between(0, 1).all() and len(X) == len(parcels)
print(X.wind_exposure.describe(percentiles=[.1, .5, .9]).round(3).to_string())
print('complet expuse:', f'{(X.wind_exposure == 1).mean():.0%}', '| complet adăpostite:', f'{(X.wind_exposure == 0).mean():.0%}')

count    17466.000
mean         0.718
std          0.373
min          0.000
10%          0.042
50%          0.951
90%          1.000
max          1.000
complet expuse: 38% | complet adăpostite: 6%
